# 📥 Download CAR - Google Colab

Este notebook permite baixar arquivos do **Cadastro Ambiental Rural (SICAR)** diretamente no Google Colab.

## 🎯 Objetivo
Permitir o download programático dos dados públicos do SICAR através de uma interface amigável no Google Colab.

## 📋 Pré-requisitos
- Conexão com internet
- Conta Google (para executar no Colab)

## 🔗 Repositório
**Projeto**: [download-car](https://github.com/Malnati/download-car)

---

## 🔧 Instalação do Pacote

Instalar o pacote `download-car` diretamente do GitHub:

In [ ]:
# Instalar dependências do sistema
!sudo apt update -qq
!sudo apt install -y tesseract-ocr tesseract-ocr-por -qq

# Instalar o pacote download-car
!pip install --quiet git+https://github.com/Malnati/download-car.git

# Instalar PaddleOCR (opcional - para usar driver Paddle)
# !pip install --quiet paddlepaddle paddleocr

print("✅ Pacote download-car instalado com sucesso!")

## 📚 Importar Bibliotecas

Importar as classes necessárias do pacote:

In [ ]:
import os
import zipfile
from datetime import datetime

# Importar classes do download-car
from download_car import DownloadCar, State, Polygon
from download_car.drivers import Tesseract
# from download_car.drivers import Paddle  # Descomente se instalar PaddleOCR

print("✅ Bibliotecas importadas com sucesso!")

## ⚙️ Configuração dos Parâmetros

Configure os parâmetros de download abaixo:

In [ ]:
# ===== CONFIGURAÇÃO DOS PARÂMETROS =====

# Estado a ser baixado
ESTADO = "SP"  # Exemplos: "SP", "RJ", "MG", "PA", "DF", etc.

# Tipo de polígono
POLIGONO = "AREA_PROPERTY"  # Exemplos: "APPS", "AREA_PROPERTY", "LEGAL_RESERVE", etc.

# Driver OCR
DRIVER = "Tesseract"  # Driver Tesseract (Paddle não está disponível na versão atual)

# Configurações de rede
TENTATIVAS = 10
TIMEOUT = 30

# Pasta de destino (será criada automaticamente)
timestamp = datetime.now().strftime("%Y%m%d_%H%MM%S")
PASTA_DESTINO = f"car_{ESTADO.lower()}_{timestamp}"

# ======================================

print("📋 Parâmetros configurados:")
print(f"   Estado: {ESTADO}")
print(f"   Polígono: {POLIGONO}")
print(f"   Driver: {DRIVER}")
print(f"   Tentativas: {TENTATIVAS}")
print(f"   Timeout: {TIMEOUT}s")
print(f"   Pasta: {PASTA_DESTINO}")

## ✅ Validação dos Parâmetros

Verificar se os parâmetros são válidos:

In [ ]:
# Validar estado
try:
    state_enum = State[ESTADO]
    print(f"✅ Estado válido: {ESTADO} ({state_enum.value})")
except KeyError:
    print(f"❌ Estado inválido: {ESTADO}")
    print("Estados disponíveis:")
    for state in State:
        print(f"  - {state.name}: {state.value}")
    raise ValueError(f"Estado '{ESTADO}' não encontrado")

# Validar polígono
try:
    polygon_enum = Polygon[POLIGONO]
    print(f"✅ Polígono válido: {POLIGONO} ({polygon_enum.value})")
except KeyError:
    print(f"❌ Polígono inválido: {POLIGONO}")
    print("Polígonos disponíveis:")
    for polygon in Polygon:
        print(f"  - {polygon.name}: {polygon.value}")
    raise ValueError(f"Polígono '{POLIGONO}' não encontrado")

# Validar driver
if DRIVER != "Tesseract":
    print(f"❌ Driver inválido: {DRIVER}")
    raise ValueError("Driver deve ser 'Tesseract' (Paddle não está disponível na versão atual)")
else:
    print(f"✅ Driver válido: {DRIVER}")

print("\n🎯 Todos os parâmetros são válidos!")
print("Pronto para iniciar o download.")

## 🚀 Executar Download

Iniciar o processo de download:

In [ ]:
# Criar pasta de destino
os.makedirs(PASTA_DESTINO, exist_ok=True)

# Selecionar driver (apenas Tesseract disponível)
driver = Tesseract

print(f"🚀 Iniciando download...")
print(f"   Estado: {ESTADO}")
print(f"   Polígono: {POLIGONO}")
print(f"   Pasta: {PASTA_DESTINO}")
print(f"   Driver: {DRIVER}")
print("\n⏳ Aguarde, o download pode levar alguns minutos...")

# Criar instância do DownloadCar
car = DownloadCar(driver=driver)

# Executar download
try:
    result = car.download_state(
        state=state_enum,
        polygon=polygon_enum,
        folder=PASTA_DESTINO,
        tries=TENTATIVAS,
        timeout=TIMEOUT
    )
    
    print("\n✅ Download concluído com sucesso!")
    print(f"📁 Arquivos salvos em: {PASTA_DESTINO}")
    
except Exception as e:
    print(f"\n❌ Erro durante o download: {str(e)}")
    raise

## 📁 Verificar Arquivos Baixados

Listar os arquivos que foram baixados:

In [ ]:
# Verificar arquivos na pasta de destino
print(f"📁 Conteúdo da pasta: {PASTA_DESTINO}")
print("=" * 50)

if os.path.exists(PASTA_DESTINO):
    files = os.listdir(PASTA_DESTINO)
    if files:
        total_size = 0
        for file in sorted(files):
            file_path = os.path.join(PASTA_DESTINO, file)
            size = os.path.getsize(file_path)
            total_size += size
            size_mb = size / (1024 * 1024)
            print(f"📄 {file} ({size_mb:.2f} MB)")
        
        total_mb = total_size / (1024 * 1024)
        print(f"\n📊 Total: {len(files)} arquivos ({total_mb:.2f} MB)")
    else:
        print("❌ Nenhum arquivo encontrado na pasta")
else:
    print(f"❌ Pasta não encontrada: {PASTA_DESTINO}")

## 📦 Compactar Arquivos

Criar um arquivo ZIP com todos os arquivos baixados:

In [ ]:
# Criar nome do arquivo ZIP
zip_filename = f"car_{ESTADO.lower()}_{timestamp}.zip"

print(f"📦 Criando arquivo ZIP: {zip_filename}")

# Criar arquivo ZIP
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    if os.path.exists(PASTA_DESTINO):
        for root, dirs, files in os.walk(PASTA_DESTINO):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, PASTA_DESTINO)
                zipf.write(file_path, arcname)
                print(f"  📄 Adicionado: {arcname}")

# Verificar tamanho do ZIP
zip_size = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\n✅ Arquivo ZIP criado: {zip_filename} ({zip_size:.2f} MB)")

## 💾 Download do Arquivo ZIP

Baixar o arquivo ZIP para o seu computador:

In [ ]:
from google.colab import files

print(f"💾 Iniciando download do arquivo: {zip_filename}")
print(f"📊 Tamanho: {zip_size:.2f} MB")
print("\n⏳ O download será iniciado automaticamente...")

# Fazer download do arquivo
files.download(zip_filename)

print("\n✅ Download iniciado! Verifique a pasta de downloads do seu navegador.")

## 🎉 Conclusão

O download foi concluído com sucesso! Os arquivos estão disponíveis:

1. **Na pasta local**: `{PASTA_DESTINO}`
2. **Como arquivo ZIP**: `{zip_filename}` (baixado automaticamente)

### 📋 Próximos Passos
- Abra o arquivo ZIP baixado
- Use os shapefiles em seu software GIS preferido (QGIS, ArcGIS, etc.)
- Para baixar outros estados ou polígonos, altere os parâmetros na seção "Configuração" e execute novamente

### 🔗 Links Úteis
- [SICAR - Sistema Nacional de Cadastro Ambiental Rural](https://www.car.gov.br/)
- [Repositório do Projeto](https://github.com/Malnati/download-car)

---

**Dúvidas?** Consulte a documentação do projeto ou abra uma issue no GitHub.